# 02 · Campaign — theozyme → scaffold → LigandMPNN (catalytic residues fixed)

**Standard slot:** *design campaign.* **For Project 18 this means:** take the theozyme, scaffold it
into many backbones (RFdiffusion2 / Riff-Diff — the **A100** step), then **LigandMPNN sequence design
fixing the catalytic residues**, and write a results CSV (D2).

Runs end-to-end on the **mock** backend with no GPU; switch to the real backends on Colab/HPC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams
Tools change. Before a campaign, HTTP-check that the pinned upstream repos still exist, and pin the
commit/tag you actually use. **RFdiffusion2 and Riff-Diff are new and move fast — VERIFY the current
public release/repo at generation time** (do not assert a repo you are unsure of); the others below
are stable enough to head-check.

In [ ]:
import requests

# Pinned upstreams (pin the COMMIT/TAG you use in env/requirements.txt + LOG.md):
STABLE_UPSTREAMS = {
    "RFdiffusion (classic motif scaffolding)": "https://github.com/RosettaCommons/RFdiffusion",
    "LigandMPNN (catalytic-residue-fixed seq design)": "https://github.com/dauparas/LigandMPNN",
    "AutoDock Vina (substrate fit)": "https://github.com/ccsb-scripps/AutoDock-Vina",
    "OpenMM (active-site MD)": "https://github.com/openmm/openmm",
}
# VERIFY-ONLY (new/fast-moving; confirm the current release before relying on a URL):
VERIFY_UPSTREAMS = [
    "RFdiffusion2 (Dauparas 2025) — VERIFY current public release/repo at generation time",
    "Riff-Diff (Schnettler 2025, Nature) — VERIFY current public release/repo at generation time",
]

for name, url in STABLE_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"[{r.status_code}] {name}\n      {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e!r}\n      {url}")
print("\nVERIFY MANUALLY (do not assert a repo URL you are unsure of):")
for v in VERIFY_UPSTREAMS:
    print("  -", v)

## 1 · Build the theozyme and scaffold it
The mock path returns placeholder backbones so the loop runs anywhere. On an A100, switch
`METHOD` to `"rfdiffusion2"` or `"riffdiff"` (verify the release) and `N_SCAFFOLDS` to 1000s.

> **A100 NOTE:** scaffolding 1000s of backbones is the compute bottleneck. Free Colab T4 can do a
> small **RFdiffusion** (classic) motif-scaffolding demo (tens of backbones); the real campaign wants
> an A100 (Colab Pro+) or HPC. The mock backend below needs no GPU at all.

In [ ]:
from enzyme_tools import build_theozyme, scaffold_motif

theo = build_theozyme("kemp_elimination")

METHOD = "mock"        # -> "rfdiffusion2" | "riffdiff" | "rfdiffusion" on Colab/HPC (verify release)
N_SCAFFOLDS = 12       # -> 1000s for the real campaign

scaffolds = scaffold_motif(theo, n=N_SCAFFOLDS, method=METHOD)
print(f"{len(scaffolds)} scaffolds via method={METHOD!r} (mock numbers are SYNTHETIC)")
print("example:", scaffolds[0])

## 2 · LigandMPNN sequence design — FIXING the catalytic residues
This is the core of enzyme sequence design: redesign the protein but **keep the catalytic residues
fixed** (and use the ligand/TS context). That is why LigandMPNN, not vanilla ProteinMPNN, is used.
On Colab set `TOOL="ligandmpnn"` (CPU-fast) and pass the fixed-positions list + ligand/TS context.

In [ ]:
from enzyme_tools import ligandmpnn_fix_catalytic

TOOL = "mock"          # -> "ligandmpnn" on Colab (CPU-fast)
SEQS_PER_BACKBONE = 4

catalytic = theo.catalytic_residue_ids()
all_designs = []
for bb in scaffolds:
    seqs = ligandmpnn_fix_catalytic(bb, catalytic, n=SEQS_PER_BACKBONE, tool=TOOL)
    for s in seqs:
        s["scaffold_id"] = bb["design_id"]
        s["scaffold_method"] = bb["method"]
        s["motif_rmsd"] = bb["motif_rmsd"]
        all_designs.append(s)
print(f"{len(all_designs)} sequences total "
      f"({len(scaffolds)} backbones x {SEQS_PER_BACKBONE}); catalytic roles fixed: {catalytic}")

## 3 · Predict + score (mock catalytic-geometry RMSD), write the results CSV
On Colab, predict each sequence with AF2/ESMFold, read the **active-site pLDDT**, and compute the
real `catalytic_geometry_rmsd` from the predicted PDB. Here the mock backend fills SYNTHETIC values
so the CSV — the input to notebook 03 — is produced anywhere.

In [ ]:
import pandas as pd
from enzyme_tools import catalytic_geometry_rmsd, dock_substrate, active_site_md

rows = []
for d in all_designs:
    # On Colab, `pred_pdb` is the AF2-predicted PDB path for this design; the mock backend keys off
    # the (non-existent) per-design path string so each design gets a DISTINCT SYNTHETIC value.
    pred_pdb = f"results/pred/{d['design_id']}.pdb"
    cg = catalytic_geometry_rmsd(pred_pdb, theo)       # mock -> SYNTHETIC (varies per design)
    dock = dock_substrate(pred_pdb, theo.substrate)    # mock -> SYNTHETIC
    md_res = active_site_md(pred_pdb, ns=10.0)         # mock -> SYNTHETIC
    # SYNTHETIC stand-ins for AF2 confidence so the plumbing runs (replace with real predictions):
    import hashlib
    h = int(hashlib.sha256(d["design_id"].encode()).hexdigest(), 16)
    plddt = 78 + (h % 20)            # 78-97, SYNTHETIC
    plddt_cat = 80 + ((h >> 7) % 18) # 80-97, SYNTHETIC
    scrmsd = round(0.8 + ((h >> 11) % 200) / 100.0, 2)  # 0.8-2.8, SYNTHETIC
    rows.append(dict(
        design_id=d["design_id"], scaffold_id=d["scaffold_id"],
        scaffold_method=d["scaffold_method"], sequence=d["sequence"],
        plddt=plddt, plddt_catalytic=plddt_cat, scrmsd=scrmsd,
        catalytic_geom_rmsd=cg, vina_score=dock["vina_score"],
        md_rmsd=md_res["md_rmsd"], synthetic=True))

camp = pd.DataFrame(rows)
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape, "(ALL NUMBERS SYNTHETIC — mock backend)")
print(f"catalytic_geom_rmsd range: {camp['catalytic_geom_rmsd'].min()}-{camp['catalytic_geom_rmsd'].max()} A")
camp.head()

## D2 checklist
- [ ] Scaffolding run logged (method, release/commit, N backbones, seed) — A100 for the real campaign.
- [ ] LigandMPNN sequences with the **catalytic residues provably fixed** (fixed-positions list logged).
- [ ] `results/campaign.csv` with one row per design (real metrics on Colab; mock here).
- [ ] Design log (every config + seed + output path) + 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared enzyme filter on `campaign.csv`.